In [1]:
# TEMPORAL PART1 — CELL 01
# RAW TIMESTAMP SOURCE DISCOVERY & READINESS
#
# READ ONLY | NO MODEL TRAINING | NO DATA MODIFICATION
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

MODEL_PATH = (
    ROOT / "04_Modeling" / "outputs"
    / "leakage_grouping"
    / "modeling_labeled_348_grouped.csv"
)

RAW_DIR = ROOT / "01_Raw Data" / "outputs" / "raw_csv"

EXPECTED_MODEL_SHA = (
    "cfa7df27079ef47bdb1b0cb526bc017aa1c8fe18dc4d91fef870a3441e2e9fb7"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

assert MODEL_PATH.is_file(), "Dataset Modeling tidak ditemukan"
assert RAW_DIR.is_dir(), "Folder raw_csv tidak ditemukan"

assert sha256_file(MODEL_PATH) == EXPECTED_MODEL_SHA, (
    "Dataset Modeling berbeda dari versi terkunci"
)

model = pd.read_csv(
    MODEL_PATH,
    encoding="utf-8-sig",
    dtype={"id": "string"},
    keep_default_na=False
)

assert len(model) == 348
assert model["id"].is_unique

assert {"id", "case", "sentiment_label"}.issubset(model.columns)

raw_files = sorted(RAW_DIR.rglob("*.csv"))

assert raw_files, "Tidak ada raw CSV ditemukan"

target_ids = set(model["id"].astype(str))

matches = []
files_with_createdat = 0
files_without_createdat = []
read_errors = []
raw_rows_scanned = 0

for path in raw_files:
    try:
        header = pd.read_csv(
            path,
            nrows=0,
            encoding="utf-8-sig"
        )

        if not {"id", "createdAt"}.issubset(header.columns):
            files_without_createdat.append(
                str(path.relative_to(ROOT))
            )
            continue

        files_with_createdat += 1

        raw = pd.read_csv(
            path,
            usecols=["id", "createdAt"],
            encoding="utf-8-sig",
            dtype={"id": "string", "createdAt": "string"},
            keep_default_na=False,
            on_bad_lines="error"
        )

        raw_rows_scanned += len(raw)

        subset = raw.loc[
            raw["id"].isin(target_ids)
        ].copy()

        if not subset.empty:
            subset["source_file"] = str(
                path.relative_to(ROOT)
            ).replace("\\", "/")

            matches.append(subset)

    except Exception as error:
        read_errors.append({
            "file": str(path.relative_to(ROOT)),
            "error": str(error)[:300]
        })

if matches:
    matched = pd.concat(matches, ignore_index=True)
else:
    matched = pd.DataFrame(
        columns=["id", "createdAt", "source_file"]
    )

matched["createdAt"] = (
    matched["createdAt"]
    .astype("string")
    .str.strip()
)

matched["parsed_utc"] = pd.to_datetime(
    matched["createdAt"],
    utc=True,
    errors="coerce"
)

matched["timestamp_valid"] = matched["parsed_utc"].notna()

# Count different VALID timestamps for each tweet ID.
valid_matches = matched.loc[matched["timestamp_valid"]]

timestamp_conflicts = (
    valid_matches.groupby("id")["parsed_utc"]
    .nunique()
)

conflict_ids = set(
    timestamp_conflicts[timestamp_conflicts > 1].index
)

valid_ids = set(valid_matches["id"].astype(str))
matched_ids = set(matched["id"].astype(str))

missing_ids = target_ids - matched_ids
matched_without_valid_time = matched_ids - valid_ids

print("=" * 72)
print("05_TEMPORAL — PART1 TIMESTAMP READINESS")
print("=" * 72)

print("Locked Modeling rows        :", len(model))
print("Unique Modeling IDs         :", model["id"].nunique())
print("Raw CSV files found         :", len(raw_files))
print("Files with createdAt        :", files_with_createdat)
print("Raw rows scanned            :", raw_rows_scanned)
print("Raw matching records        :", len(matched))
print("Unique matched IDs          :", len(matched_ids))
print("Unique valid timestamp IDs  :", len(valid_ids))
print("Missing raw IDs             :", len(missing_ids))
print("Matched IDs no valid time   :", len(matched_without_valid_time))
print("Conflicting timestamp IDs   :", len(conflict_ids))
print("Raw file read errors        :", len(read_errors))

print("\nMODEL DISTRIBUTION BY CASE")
display(
    model.groupby("case")
    .agg(
        tweet_count=("id", "nunique")
    )
    .reset_index()
)

print("\nTIMESTAMP COVERAGE BY CASE")
coverage = model[["id", "case"]].copy()

coverage["found_raw"] = coverage["id"].isin(matched_ids)
coverage["valid_timestamp"] = coverage["id"].isin(valid_ids)
coverage["timestamp_conflict"] = coverage["id"].isin(conflict_ids)

display(
    coverage.groupby("case")
    .agg(
        tweets=("id", "count"),
        found_raw=("found_raw", "sum"),
        valid_timestamp=("valid_timestamp", "sum"),
        conflicting_timestamp=("timestamp_conflict", "sum")
    )
    .reset_index()
)

if not valid_matches.empty:
    print("\nEARLIEST / LATEST RAW TIMESTAMPS (UTC)")
    print("Earliest:", valid_matches["parsed_utc"].min())
    print("Latest  :", valid_matches["parsed_utc"].max())

    first_valid = (
        valid_matches.sort_values(["id", "parsed_utc"])
        .drop_duplicates("id")
        [["id", "parsed_utc"]]
    )

    date_preview = model[["id", "case"]].merge(
        first_valid,
        on="id",
        how="left",
        validate="one_to_one"
    )

    date_preview["date_wib"] = (
        date_preview["parsed_utc"]
        .dt.tz_convert("Asia/Jakarta")
        .dt.date
    )

    print("\nDATE COVERAGE BY CASE (WIB)")
    display(
        date_preview.groupby("case")
        .agg(
            earliest=("date_wib", "min"),
            latest=("date_wib", "max"),
            rows_with_dates=("parsed_utc", "count")
        )
        .reset_index()
    )

if read_errors:
    print("\nRAW FILE READ ERRORS (FIRST 10)")
    display(pd.DataFrame(read_errors).head(10))

if files_without_createdat:
    print("\nFILES WITHOUT TIMESTAMP COLUMN (FIRST 10)")
    print("\n".join(files_without_createdat[:10]))

print("\nFINAL PREFLIGHT DECISION")

if (
    len(valid_ids) == 348
    and len(conflict_ids) == 0
    and len(read_errors) == 0
):
    print("TEMPORAL TIMESTAMP PREFLIGHT: PASS")
else:
    print("TEMPORAL TIMESTAMP PREFLIGHT: REVIEW REQUIRED")

print("READ-ONLY SOURCE SCAN COMPLETE")
print("=" * 72)


05_TEMPORAL — PART1 TIMESTAMP READINESS
Locked Modeling rows        : 348
Unique Modeling IDs         : 348
Raw CSV files found         : 65
Files with createdAt        : 57
Raw rows scanned            : 1388
Raw matching records        : 831
Unique matched IDs          : 348
Unique valid timestamp IDs  : 348
Missing raw IDs             : 0
Matched IDs no valid time   : 0
Conflicting timestamp IDs   : 0
Raw file read errors        : 0

MODEL DISTRIBUTION BY CASE


C:\Users\Yusak SR\AppData\Local\Temp\ipykernel_1372\4237529174.py:124: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  matched["parsed_utc"] = pd.to_datetime(


,case,tweet_count
0,lombok_2018,101
1,palu_2018,105
2,sumatra_2025,142



TIMESTAMP COVERAGE BY CASE


,case,tweets,found_raw,valid_timestamp,conflicting_timestamp
0,lombok_2018,101,101,101,0
1,palu_2018,105,105,105,0
2,sumatra_2025,142,142,142,0



EARLIEST / LATEST RAW TIMESTAMPS (UTC)
Earliest: 2018-08-06 07:40:02+00:00
Latest  : 2025-12-04 14:43:34+00:00

DATE COVERAGE BY CASE (WIB)


,case,earliest,latest,rows_with_dates
0,lombok_2018,2018-08-06,2018-08-14,101
1,palu_2018,2018-09-29,2018-10-03,105
2,sumatra_2025,2025-11-29,2025-12-04,142



FILES WITHOUT TIMESTAMP COLUMN (FIRST 10)
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_539338a38118f5470d1b.csv
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_7085e181db6c8ff0911a.csv
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_971bf25151d9f4419d91.csv
01_Raw Data\outputs\raw_csv\palu_2018\palu_2018_0ac1c9ee684c47c748ff.csv
01_Raw Data\outputs\raw_csv\palu_2018\palu_2018_b4187151cd0d81be2864.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_04ed36216ec543478f4a.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_4f9a9ad2f4ca2fcd88d6.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_a2a75c0483636c3341eb.csv

FINAL PREFLIGHT DECISION
TEMPORAL TIMESTAMP PREFLIGHT: PASS
READ-ONLY SOURCE SCAN COMPLETE
